In [ ]:
# riss.kr 에서 특정 키워드로 논문 / 학술 자료 검색하기

# Step 1. 필요한 모듈을 로딩합니다
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
import time
import math
from bs4 import BeautifulSoup
import pandas as pd

# Step 2. 사용자에게 검색 관련 정보들을 입력 받습니다.
print("=" *100)
print(" 이 크롤러는 RISS 사이트의 논문 및 학술자료 수집용 웹크롤러입니다.")
print("=" *100)
query_txt = input('1.수집할 자료의 키워드는 무엇입니까? : ')

# Step 3. 수집된 데이터를 저장할 파일 이름 입력받기
ft_name = input('2.결과를 저장할 txt형식의 파일명을 쓰세요(예: c:\\py_temp\\riss.txt): ')
fc_name = input('3.결과를 저장할 csv형식의 파일명을 쓰세요(예: c:\\py_temp\\riss.csv): ')
fx_name = input('4.결과를 저장할 xlsx형식의 파일명을 쓰세요(예: c:\\py_temp\\riss.xlsx): ')

# Step 4. 크롬 드라이버 설정 및 웹 페이지 열기
s = Service("c:/py_temp/chromedriver.exe")
driver = webdriver.Chrome(service=s)

url = 'https://www.riss.kr/'
driver.get(url)
time.sleep(3)
driver.maximize_window()

# Step 5. 검색어 입력 후 조회
element = driver.find_element(By.ID,'query')
element.click()
element.send_keys(query_txt)
element.send_keys("\n")

# Step 6. 학위논문 선택
driver.find_element(By.LINK_TEXT,'학위논문').click()
time.sleep(2)

# Step 7. BeautifulSoup 로 현재 페이지 파싱
html_1 = driver.page_source
soup_1 = BeautifulSoup(html_1, 'html.parser')

# Step 8. 총 검색 건수 확인
total_cnt = soup_1.find('span','num').get_text()
print('키워드 %s (으)로 총 %s 건의 학위논문이 검색되었습니다' %(query_txt,total_cnt))

collect_cnt = int(input('이 중에서 몇 건을 수집하시겠습니까?: '))
collect_page_cnt = math.ceil(collect_cnt / 10)

print('%s 건을 수집하기 위해 %s 페이지를 조회합니다.' %(collect_cnt,collect_page_cnt))
print('=' *80)

# Step 9. 데이터 저장용 리스트 생성
no2 = []
title2 = []
writer2 = []
org2 = []
year2 = []
degree2 = []
url2 = []

no = 1

for a in range(1, collect_page_cnt + 1):

    html_2 = driver.page_source
    soup_2 = BeautifulSoup(html_2, 'html.parser')

    content_2 = soup_2.find('div','srchResultListW').find_all('li')

    for b in content_2:
        try:
            cont = b.find('div','cont')
            title_tag = cont.find('p','title').find('a')
            title = title_tag.get_text().strip()
        except:
            continue
        else:
            f = open(ft_name, 'a', encoding="UTF-8")

            # 번호
            print('1.번호:',no)
            no2.append(no)
            f.write('\n1.번호:' + str(no))

            # 제목
            print('2.논문제목:',title)
            title2.append(title)
            f.write('\n2.논문제목:' + title)

            # 저자
            writer = cont.find('span','writer').get_text().strip()
            print('3.저자:',writer)
            writer2.append(writer)
            f.write('\n3.저자:' + writer)

            # 소속기관
            org = cont.find('span','assigned').get_text().strip()
            print('4.소속기관:',org)
            org2.append(org)
            f.write('\n4.소속기관:' + org)

            # 발행년도 + 논문집
            etc = cont.find('p','etc')
            spans = etc.find_all('span')

            year = spans[-2].get_text().strip()
            degree = spans[-1].get_text().strip()

            org = cont.find('span','assigned').get_text().strip()
            
            print('5.발행년도:',year)
            year2.append(year)
            f.write('\n5.발행년도:' + year)

            
            print('5.발행년도:',year)
            year2.append(year)
            f.write('\n5.발행년도:' + year)

            print('6.논문집:',degree)
            degree2.append(degree)
            f.write('\n6.논문집:' + degree)

            # URL
            url= cont.find('p','title').find('a')['href']            
            print('7.URL:',url)
            url2.append(url)
            f.write('\n7.URL:' + url+ '\n')

            f.close()

            no += 1
            print("\n")

            if no > collect_cnt:
                break

            time.sleep(1)

    if no > collect_cnt:
        break

    # 페이지 이동
    next_page = a + 1
    time.sleep(1)

    try:
        driver.find_element(By.LINK_TEXT, str(next_page)).click()
    except:
        try:
            driver.find_element(By.LINK_TEXT,('다음 페이지로')).click()
        except:
            print("마지막 페이지입니다.")
            break

    time.sleep(2)

# Step 10. DataFrame 생성 및 저장
df = pd.DataFrame()
df['번호'] = no2
df['제목'] = title2
df['저자'] = writer2
df['소속(발행)기관'] = org2
df['발행년도'] = year2
df['논문집'] = degree2
df['URL'] = url2

# 엑셀 저장
df.to_excel(fx_name, index=False, engine='openpyxl')

# CSV 저장
df.to_csv(fc_name, index=False, encoding="utf-8-sig")

print('요청하신 데이터 수집 작업이 정상적으로 완료되었습니다')
driver.quit()

In [ ]:
# riss.kr 에서 특정 키워드로 논문 / 학술 자료 검색하기
#Step 1. 필요한 모듈을 로딩합니다
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.service import Service
import time

#Step 2. 사용자에게 검색 관련 정보들을 입력 받습니다.
print("=" *100)
print(" 이 크롤러는 RISS 사이트의 논문 및 학술자료 수집용 웹크롤러입니다.")
print("=" *100)
query_txt = input('1.수집할 자료의 키워드는 무엇입니까? : ')

#Step 3. 수집된 데이터를 저장할 파일 이름 입력받기
ft_name = input('2.결과를 저장할 txt형식의 파일명을 쓰세요(예: c:\\py_temp\\riss.txt): ')
fc_name = input('3.결과를 저장할 csv형식의 파일명을 쓰세요(예: c:\\py_temp\\riss.csv): ')
fx_name = input('4.결과를 저장할 xlsx형식의 파일명을 쓰세요(예: c:\\py_temp\\riss.xlsx): ')

#Step 4. 크롬 드라이버 설정 및 웹 페이지 열기
s = Service("c:/py_temp/chromedriver.exe")
driver = webdriver.Chrome(service=s)

url = 'https://www.riss.kr/'
driver.get(url)
time.sleep(5)
driver.maximize_window()

#Step 5. 자동으로 검색어 입력 후 조회하기
element = driver.find_element(By.ID,'query')
driver.find_element(By.ID,'query').click( )
element.send_keys(query_txt)
element.send_keys("\n")

#Step 6.학위 논문 선택하기
driver.find_element(By.LINK_TEXT,'학위논문').click()
time.sleep(2)

#Step 7.Beautiful Soup 로 본문 내용만 추출하기
from bs4 import BeautifulSoup
html_1 = driver.page_source
soup_1 = BeautifulSoup(html_1, 'html.parser')

content_1 = soup_1.find('div','srchResultListW').find_all('li')
for i in content_1 :
    print(i.get_text().replace("\n",""))

#Step 8. 총 검색 건수를 보여주고 수집할 건수 입력받기
import math
total_cnt = soup_1.find('div','searchBox pd').find('span','num').get_text()
print('키워드 %s (으)로 총 %s 건의 학위논문이 검색되었습니다' %(query_txt,total_cnt))
collect_cnt = int(input('이 중에서 몇 건을 수집하시겠습니까?: '))
collect_page_cnt = math.ceil(collect_cnt / 10)
print('%s 건을 수집하기 위해 %s 페이지를 조회합니다.' %(collect_cnt,collect_page_cnt))
print('=' *80)

#Step 9. 각 항목별로 데이터를 추출하여 리스트에 저장하기
no2 = [ ] #번호 저장
title2 = [ ] #논문제목 저장
writer2 = [ ] #논문저자 저장
org2 = [ ] #소속기관 저장
no = 1



for a in range(1, collect_page_cnt + 1) :
    
    html_2 = driver.page_source
    soup_2 = BeautifulSoup(html_2, 'html.parser')

    content_2 = soup_2.find('div','srchResultListW').find_all('li')
    
    for b in content_2 :
        #1. 논문제목 있을 경우만
        try :
            title = b.find('div','cont').find('p','title').get_text()
        except :
            continue # 논문제목 못찾으면 오류 발생, 해당 항목 건너뜀 
        else :
            f = open(ft_name, 'a' , encoding="UTF-8") #append(추가모드), 기존 파일에 계속 추가 
            print('1.번호:',no)
            no2.append(no)
            f.write('\n'+'1.번호:' + str(no))

            print('2.논문제목:',title)
            title2.append(title)
            f.write('\n' + '2.논문제목:' + title)

            writer = b.find('span','writer').get_text()
            print('3.저자:',writer)
            writer2.append(writer)
            f.write('\n' + '3.저자:' + writer)

            org = b.find('span','assigned').get_text()
            print('4.소속기관:' , org)
            org2.append(org)
            f.write('\n' + '4.소속기관:' + org + '\n')

            f.close( )

            no += 1
            print("\n")

            
            if no > collect_cnt :
                break

            time.sleep(1) 

    a += 1
    b = str(a)
            

    time.sleep(1) # 페이지 변경 전 1초 대기

  
    try :
        driver.find_element(By.LINK_TEXT ,'%s' %b).click()
    except :
        driver.find_element(By.LINK_TEXT,('다음 페이지로')).click()
        print("요청하신 작업이 모두 완료되었습니다")
    time.sleep(1) # 페이지 변경 전 1초 대기



# Step 10. 수집된 데이터를 xlsx와 csv 형태로 저장하기
import pandas as pd

df = pd.DataFrame()
df['번호']=no2
df['제목']=pd.Series(title2)
df['저자']=pd.Series(writer2)
df['소속(발행)기관']=pd.Series(org2)

# xls 형태로 저장하기
df.to_excel(fx_name, index=False,  engine='openpyxl')

#encoding="utf-8" ,

# csv 형태로 저장하기
df.to_csv(fc_name, index=False, encoding="utf-8-sig")
print('요청하신 데이터 수집 작업이 정상적으로 완료되었습니다')